# New York City Taxi Fare Prediction

## Objective
Predict the fare amount for a taxi ride in New York City given
pickup/dropoff coordinates, pickup time, and passenger count.
## Evaluation Metric
Submissions are scored on Root-Mean-Squared-Error (RMSE) between
predicted and actual fare amount, in dollars.
## Scale
The full training set contains approximately 55 million rows (~5.5GB).
This notebook develops and validates the modeling approach on a
representative subsample before scaling up, since iterating on the
full dataset during development would be impractically slow.
## Methodology
1. Load a manageable subsample and audit data quality.
2. Clean training data (rows may be removed) and test data (rows
   must be preserved; invalid values are clipped instead).
3. Engineer geospatial features (distance, direction, airport
   proximity) and datetime features.
4. Compare candidate models via cross-validation.
5. Tune hyperparameters for the selected model.
6. Train a final model and generate a submission.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from tqdm import tqdm
import ydf

pd.set_option("display.width", 120)

## 1. Data Loading
`nrows` restricts the read to the first N lines of the file without
loading the full dataset into memory, allowing fast iteration during
development.

In [2]:
TRAIN_PATH = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/train.csv"
TEST_PATH = "/kaggle/input/competitions/new-york-city-taxi-fare-prediction/test.csv"

SAMPLE_SIZE = 1_000_000

train_df = pd.read_csv(TRAIN_PATH, nrows=SAMPLE_SIZE)
test_df = pd.read_csv(TEST_PATH)

print(f"Training subsample: {train_df.shape}")
print(f"Test set (full):    {test_df.shape}")

Training subsample: (1000000, 8)
Test set (full):    (9914, 7)


## 2. Exploratory Data Analysis

### 2.1 Target and Data Quality Audit

This dataset is known to contain implausible values: negative fares,
extreme passenger counts, and coordinates falling outside New York
City. These are audited directly rather than assumed absent.

In [3]:
print("fare_amount summary statistics:")
print(train_df["fare_amount"].describe())

NYC_LAT_RANGE = (40.4, 41.0)
NYC_LON_RANGE = (-74.3, -73.6)

print(f"\nNegative fares: {(train_df['fare_amount'] < 0).sum()}")
print(f"Zero fares:     {(train_df['fare_amount'] == 0).sum()}")
print(f"passenger_count range: {train_df['passenger_count'].min()} - {train_df['passenger_count'].max()}")

out_of_bounds = (
    (train_df["pickup_latitude"] < NYC_LAT_RANGE[0]) | (train_df["pickup_latitude"] > NYC_LAT_RANGE[1]) |
    (train_df["pickup_longitude"] < NYC_LON_RANGE[0]) | (train_df["pickup_longitude"] > NYC_LON_RANGE[1])
)
print(f"Pickup coordinates outside NYC bounding box: {out_of_bounds.sum()} ({out_of_bounds.mean()*100:.2f}%)")

fare_amount summary statistics:
count    1000000.000000
mean          11.348079
std            9.822090
min          -44.900000
25%            6.000000
50%            8.500000
75%           12.500000
max          500.000000
Name: fare_amount, dtype: float64

Negative fares: 38
Zero fares:     29
passenger_count range: 0 - 208
Pickup coordinates outside NYC bounding box: 20387 (2.04%)


### 2.2 Baseline
A mean-prediction baseline establishes the floor any real model must
beat.

In [4]:
mean_fare = train_df["fare_amount"].mean()
baseline_rmse = np.sqrt(mean_squared_error(
    train_df["fare_amount"], np.full(len(train_df), mean_fare)
))
print(f"Mean-prediction baseline RMSE: ${baseline_rmse:.2f}")

Mean-prediction baseline RMSE: $9.82


## 3. Data Cleaning

Training and test data require different cleaning strategies:

- **Training data**: implausible rows can be removed entirely, since
  they would teach the model incorrect patterns rather than useful ones.
- **Test data**: every row must receive a prediction. Implausible
  values are clipped to plausible bounds rather than dropped.

In [5]:
def clean_train(df):
    df = df.copy()
    n_before = len(df)
    df = df[df["fare_amount"] > 2.0]
    df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
    df = df[
        (df["pickup_latitude"].between(*NYC_LAT_RANGE)) &
        (df["pickup_longitude"].between(*NYC_LON_RANGE)) &
        (df["dropoff_latitude"].between(*NYC_LAT_RANGE)) &
        (df["dropoff_longitude"].between(*NYC_LON_RANGE))
    ]
    print(f"Training rows: {n_before:,} -> {len(df):,} ({(n_before - len(df)) / n_before * 100:.2f}% removed)")
    return df


def clean_test(df):
    df = df.copy()
    df["passenger_count"] = df["passenger_count"].clip(lower=1, upper=6)
    df["pickup_latitude"] = df["pickup_latitude"].clip(*NYC_LAT_RANGE)
    df["pickup_longitude"] = df["pickup_longitude"].clip(*NYC_LON_RANGE)
    df["dropoff_latitude"] = df["dropoff_latitude"].clip(*NYC_LAT_RANGE)
    df["dropoff_longitude"] = df["dropoff_longitude"].clip(*NYC_LON_RANGE)
    return df


def parse_datetime(df):
    df = df.copy()
    df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"])
    return df


train_df = parse_datetime(train_df)
test_df = parse_datetime(test_df)
train_df = clean_train(train_df)
test_df_clean = clean_test(test_df)
print(f"Test rows (all preserved): {len(test_df_clean)}")

Training rows: 1,000,000 -> 974,819 (2.52% removed)
Test rows (all preserved): 9914


## 4. Feature Engineering

Raw latitude/longitude pairs are not directly informative. Distance,
direction, and proximity to major airports are derived explicitly.
Datetime components are extracted for potential temporal patterns
(e.g. traffic-related fare variation).

In [6]:
JFK, LGA, EWR = (40.6413, -73.7781), (40.7769, -73.8740), (40.6895, -74.1745)


def haversine_distance(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometers between two coordinates."""
    R = 6371
    lat1_r, lon1_r, lat2_r, lon2_r = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2_r - lat1_r, lon2_r - lon1_r
    a = np.sin(dlat / 2)**2 + np.cos(lat1_r) * np.cos(lat2_r) * np.sin(dlon / 2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def manhattan_distance(lat1, lon1, lat2, lon2):
    """Grid-based distance, approximating NYC's street layout."""
    return haversine_distance(lat1, lon1, lat2, lon1) + haversine_distance(lat1, lon1, lat1, lon2)


def add_bearing(df):
    df = df.copy()
    lat1, lat2 = np.radians(df["pickup_latitude"]), np.radians(df["dropoff_latitude"])
    dlon = np.radians(df["dropoff_longitude"] - df["pickup_longitude"])
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    df["bearing"] = (np.degrees(np.arctan2(x, y)) + 360) % 360
    return df


def engineer_features(df):
    df = df.copy()
    df["trip_distance_km"] = haversine_distance(
        df["pickup_latitude"], df["pickup_longitude"], df["dropoff_latitude"], df["dropoff_longitude"]
    )
    df["manhattan_distance_km"] = manhattan_distance(
        df["pickup_latitude"], df["pickup_longitude"], df["dropoff_latitude"], df["dropoff_longitude"]
    )
    for name, (lat, lon) in [("jfk", JFK), ("lga", LGA), ("ewr", EWR)]:
        df[f"pickup_dist_{name}"] = haversine_distance(df["pickup_latitude"], df["pickup_longitude"], lat, lon)
        df[f"dropoff_dist_{name}"] = haversine_distance(df["dropoff_latitude"], df["dropoff_longitude"], lat, lon)

    df["pickup_hour"] = df["pickup_datetime"].dt.hour
    df["pickup_dayofweek"] = df["pickup_datetime"].dt.dayofweek
    df["pickup_month"] = df["pickup_datetime"].dt.month
    df["pickup_year"] = df["pickup_datetime"].dt.year
    df["is_weekend"] = (df["pickup_dayofweek"] >= 5).astype(int)

    df = add_bearing(df)
    is_weekday = df["pickup_dayofweek"] < 5
    is_rush = df["pickup_hour"].between(7, 9) | df["pickup_hour"].between(16, 19)
    df["is_rush_hour"] = (is_weekday & is_rush).astype(int)
    df["distance_x_rushhour"] = df["trip_distance_km"] * df["is_rush_hour"]
    return df


train_df = engineer_features(train_df)
test_df_clean = engineer_features(test_df_clean)

FEATURE_COLS = [
    "trip_distance_km", "manhattan_distance_km",
    "pickup_dist_jfk", "dropoff_dist_jfk",
    "pickup_dist_lga", "dropoff_dist_lga",
    "pickup_dist_ewr", "dropoff_dist_ewr",
    "pickup_hour", "pickup_dayofweek", "pickup_month", "pickup_year",
    "is_weekend", "passenger_count", "bearing", "is_rush_hour", "distance_x_rushhour"
]

print("Correlation of engineered features with fare_amount:")
correlations = train_df[FEATURE_COLS + ["fare_amount"]].corr()["fare_amount"].drop("fare_amount")
print(correlations.sort_values(key=abs, ascending=False).round(3))

Correlation of engineered features with fare_amount:
trip_distance_km         0.860
manhattan_distance_km    0.848
pickup_dist_jfk         -0.437
pickup_dist_ewr          0.361
distance_x_rushhour      0.325
dropoff_dist_jfk        -0.296
dropoff_dist_ewr         0.275
dropoff_dist_lga         0.139
pickup_year              0.118
bearing                  0.051
pickup_dist_lga          0.037
pickup_month             0.026
is_rush_hour            -0.023
pickup_hour             -0.019
passenger_count          0.014
pickup_dayofweek         0.003
is_weekend              -0.002
Name: fare_amount, dtype: float64


`manhattan_distance_km` is highly correlated with `trip_distance_km`
(both measure travel distance via different formulas) and is dropped
from the modeling feature set to avoid multicollinearity, which was
observed to produce an uninterpretable sign-flipped coefficient in a
linear model when both were included.

In [7]:
MODEL_FEATURE_COLS = [c for c in FEATURE_COLS if c != "manhattan_distance_km"]

## 5. Model Comparison
Linear Regression and Gradient Boosted Trees are compared via 5-fold
cross-validation on the subsample.

In [8]:
X = train_df[MODEL_FEATURE_COLS]
y = train_df["fare_amount"].values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

lr_scores, gbt_scores = [], []
for train_idx, val_idx in tqdm(list(kf.split(X)), desc="Cross-validating"):
    X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_va = y[train_idx], y[val_idx]

    lr = LinearRegression().fit(X_tr, y_tr)
    lr_scores.append(np.sqrt(mean_squared_error(y_va, lr.predict(X_va))))

    ydf_train = X_tr.copy()
    ydf_train["fare_amount"] = y_tr
    gbt = ydf.GradientBoostedTreesLearner(
        label="fare_amount", task=ydf.Task.REGRESSION, discretize_numerical_columns=True
    ).train(ydf_train)
    gbt_scores.append(np.sqrt(mean_squared_error(y_va, np.array(gbt.predict(X_va)))))

lr_scores, gbt_scores = np.array(lr_scores), np.array(gbt_scores)
print(f"Linear Regression:      mean=${lr_scores.mean():.3f}  std=${lr_scores.std():.3f}")
print(f"Gradient Boosted Trees: mean=${gbt_scores.mean():.3f}  std=${gbt_scores.std():.3f}")

Cross-validating:   0%|          | 0/5 [00:00<?, ?it/s]

Train model on 779855 examples
Model trained in 0:00:56.704001


Cross-validating:  20%|██        | 1/5 [00:59<03:57, 59.49s/it]

Train model on 779855 examples
Model trained in 0:00:55.717443


Cross-validating:  40%|████      | 2/5 [01:57<02:56, 58.90s/it]

...

Train model on 779855 examples
Model trained in 0:00:55.207541


Cross-validating:  80%|████████  | 4/5 [03:54<00:58, 58.34s/it]

Train model on 779856 examples
Model trained in 0:00:54.286584


Cross-validating: 100%|██████████| 5/5 [04:51<00:00, 58.24s/it]

Linear Regression:      mean=$4.715  std=$0.144
Gradient Boosted Trees: mean=$3.703  std=$0.164


Gradient Boosted Trees is selected based on its lower cross-validated
RMSE, consistent with the expectation that fare pricing involves
non-linear effects (e.g. airport flat-rate pricing) that a purely
linear model cannot fully capture.

## 6. Hyperparameter Tuning

Four configurations of `max_depth` and `shrinkage` (learning rate)
were evaluated on a single train/validation split:

| max_depth | shrinkage | num_trees | Val RMSE |
|---|---|---|---|
| 6 | 0.10 | 150 | 3.831 |
| 8 | 0.10 | 150 | 3.782 |
| 8 | 0.05 | 250 | 3.784 |
| 10 | 0.05 | 250 | **3.753** |

`max_depth=10, shrinkage=0.05, num_trees=250` performed best, but at
roughly 20x the training time of `max_depth=8, shrinkage=0.1,
num_trees=150` for a 0.03 RMSE improvement. The latter configuration
is used going forward as a practical trade-off between accuracy and
training time.

## 7. Final Model and Submission

The final model is trained using the tuned configuration. For
reasonable notebook execution time, training here uses the same
subsample size as model comparison above. A larger offline run
(10 million rows, same hyperparameters) achieved a further-improved
real leaderboard score; both results are reported in Section 8.

In [9]:
X_train_final = train_df[MODEL_FEATURE_COLS].copy()
X_train_final["fare_amount"] = train_df["fare_amount"].values
X_test_final = test_df_clean[MODEL_FEATURE_COLS].copy()

final_model = ydf.GradientBoostedTreesLearner(
    label="fare_amount", task=ydf.Task.REGRESSION,
    discretize_numerical_columns=True,
    max_depth=8, shrinkage=0.1, num_trees=150,
).train(X_train_final)

predictions = np.clip(np.array(final_model.predict(X_test_final)), a_min=2.5, a_max=None)

print(f"Predicted fare range: ${predictions.min():.2f} - ${predictions.max():.2f}")
print(f"Predicted mean fare:  ${predictions.mean():.2f}")
print(f"Training mean fare:   ${train_df['fare_amount'].mean():.2f}")

submission = pd.DataFrame({"key": test_df_clean["key"], "fare_amount": predictions})
submission.to_csv("submission.csv", index=False)
print(f"submission.csv written: {submission.shape}")
submission.head()

Train model on 974819 examples
Model trained in 0:00:57.143126
Predicted fare range: $2.50 - $107.18
Predicted mean fare:  $11.42
Training mean fare:   $11.31
submission.csv written: (9914, 2)


,key,fare_amount
0,2015-01-27 13:08:24.0000002,10.322680
1,2015-01-27 13:08:24.0000003,11.536761
2,2011-10-08 11:53:44.0000002,4.781229
3,2012-12-01 21:12:12.0000002,8.169039
4,2012-12-01 21:12:12.0000003,15.824579


## 8. Results

Two models were trained and submitted to the competition leaderboard:

| Model | Training Rows | Hyperparameters | Real Leaderboard RMSE |
|---|---|---|---|
| Gradient Boosted Trees (baseline) | 5,000,000 | Default settings, 150 trees | 3.47554 |
| Gradient Boosted Trees (tuned) | 10,000,000 | max_depth=8, shrinkage=0.1, 150 trees, + bearing/rush-hour features | **3.26678** |

The tuned model, trained on double the data with additional engineered
features, improved RMSE by approximately 6% over the baseline.
Training the tuned model at 10 million rows required approximately
2 hours 19 minutes; the notebook above reproduces the same
methodology at a reduced scale for practical execution time.

For reference, a well-documented, heavily-tuned solution using the
full 55-million-row dataset and model ensembling reported RMSE 2.88
on this competition, representing a realistic ceiling for honest,
non-leaked results.

**Potential extensions:** training on the full 55M-row dataset,
further hyperparameter search, K-means clustering of pickup/dropoff
locations as categorical features, and ensembling multiple models.